In [1]:
from azureml.core import Workspace

ws = Workspace.from_config()
datastore = ws.get_default_datastore()

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("premier_league_features.csv")

feature_cols = ["home_form", "away_form", "home_goal_diff_trend", "away_goal_diff_trend", "h2h_home_points"]
model_cols = feature_cols + ["target"]

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx][model_cols] 
test_df = df.iloc[split_idx:][model_cols]   

print(f"Train: {len(train_df)} sor, Test: {len(test_df)} sor")

train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)

Train: 1204 sor, Test: 301 sor


In [ ]:
mltable_content = """
paths:
  - file: ./train_data.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: 'utf8'
      header: all_files_same_headers
"""

import os
os.makedirs("train_mltable", exist_ok=True)

with open("train_mltable/MLTable", "w") as f:
    f.write(mltable_content)

import shutil
shutil.copy("train_data.csv", "train_mltable/train_data.csv")

print("MLTable mappa létrehozva")

MLTable mappa létrehozva


In [4]:
datastore.upload(
    src_dir="train_mltable",
    target_path="football-data/train_mltable/",
    overwrite=True
)
print("MLTable feltöltve")

Uploading an estimated of 2 files
Uploading train_mltable/MLTable
Uploaded train_mltable/MLTable, 1 files out of an estimated total of 2
Uploading train_mltable/train_data.csv
Uploaded train_mltable/train_data.csv, 2 files out of an estimated total of 2
Uploaded 2 files
MLTable feltöltve


In [5]:
from azure.ai.ml import MLClient, Input, automl
from azure.ai.ml.constants import AssetTypes

classification_job = automl.classification(
    training_data=Input(
        type=AssetTypes.MLTABLE,
        path="azureml://datastores/workspaceblobstore/paths/football-data/train_mltable/"
    ),
    target_column_name="target",
    primary_metric="accuracy",
    compute="molnarbotondkristof1",
    experiment_name="football-prediction-automl",
    n_cross_validations=5,
)

classification_job.set_limits(
    timeout_minutes=20,
    trial_timeout_minutes=10,
    max_trials=15,
    enable_early_termination=True,
)

print("AutoML job konfigurálva (MLTable-lel)")

AutoML job konfigurálva (MLTable-lel)


In [ ]:
returned_job = ml_client.jobs.create_or_update(classification_job)
print(f"Job elindítva: {returned_job.name}")
print(f"Studio link: {returned_job.studio_url}")